# Final Model V2.4 - Cross-Attention Fusion (Spatial-Temporal)

- **Base**: `Final_env_v2_3.ipynb` (Gated Fusion Success)
- **Improvement**: **Cross-Attention Fusion**
- **Concept**:
    - GRU provides the **Intent** (Query) -> "Player turning left"
    - CNN provides the **Spatial Map** (Key/Value) -> "Open space at (x,y)"
    - **Cross-Attention**: The model dynamically attends to specific spatial regions based on the player's movement intent.
- **Preprocessor**: `Preprocessing_final_v2` (Pure Physics, No Angle)

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from sklearn.model_selection import GroupKFold
import cv2 
from tqdm import tqdm
import joblib
import json

# [Same Preprocessor as V2.3]
from Preprocessing_final_v2_fixed import FootballPreprocessorMultimodal

FOLD_SEEDS = {
    1: 42, 2: 43, 3: 44, 4: 45, 5: 46,
    6: 47, 7: 48, 8: 49, 9: 50, 10: 51,
}

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

Device: cuda


## 1. 데이터 로드 및 전처리 (V2.3 동일)

In [2]:
DATA_DIR = './data'
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
print(f"Loaded {len(train_df)} rows")

Loaded 356721 rows


In [3]:
# Dataset Code - Identical to V2.3
class MultiModalDataset(Dataset):
    def __init__(self, episodes, img_size=(68, 105), augment=False, cache_images=True):
        self.episodes = episodes
        self.H, self.W = img_size
        self.augment = augment
        self.cache_images = cache_images
        self.img_cache = None
        
        if self.cache_images:
            print(f"Pre-rendering {len(episodes)} images (Cache Enabled)...")
            self.img_cache = [self._generate_image(ep['cont']) for ep in tqdm(episodes, desc="Caching Images")]

    def __len__(self):
        return len(self.episodes)

    def _generate_image(self, cont_data):
        img_np = np.zeros((2, self.H, self.W), dtype=np.float32)
        if len(cont_data) == 0:
             return torch.tensor(img_np, dtype=torch.float32)

        seq_len = cont_data.shape[0]
        xs = cont_data[:, 0]
        ys = cont_data[:, 1]
        
        x_idxs = (xs * (self.W - 1)).round().astype(int).clip(0, self.W - 1)
        y_idxs = (ys * (self.H - 1)).round().astype(int).clip(0, self.H - 1)
        
        for t in range(seq_len):
            px, py = x_idxs[t], y_idxs[t]
            img_np[0, py, px] = 1.0
            decay_val = (t + 1) / seq_len
            if t > 0:
                prev_px, prev_py = x_idxs[t-1], y_idxs[t-1]
                cv2.line(img_np[1], (prev_px, prev_py), (px, py), color=decay_val, thickness=1)
            else:
                img_np[1, py, px] = decay_val
        return torch.tensor(img_np, dtype=torch.float32)

    def __getitem__(self, idx):
        data = self.episodes[idx]
        cont = data['cont'].copy()
        target = data['target'].copy()
        
        if self.augment and random.random() < 0.5:
            cont[:, 1] = 1.0 - cont[:, 1] # start_y
            cont[:, 3] = 1.0 - cont[:, 3] # end_y_prev
            cont[:, 5] = -cont[:, 5]      # dy_prev
            target[1] = 1.0 - target[1]
            if self.cache_images:
                base_img = self.img_cache[idx]
                img = torch.flip(base_img, [1])
            else:
                img = self._generate_image(cont)
        else:
            if self.cache_images:
                img = self.img_cache[idx]
            else:
                img = self._generate_image(cont)
        
        cont_tensor = torch.tensor(cont, dtype=torch.float32)
        cat_tensor = torch.tensor(data['cat'], dtype=torch.long)
        target_tensor = torch.tensor(target, dtype=torch.float32)
        
        seq_len = cont.shape[0]
        aux_target = np.zeros((seq_len, 2), dtype=np.float32)
        if seq_len > 1:
            aux_target[:-1] = cont[1:, 0:2]
        aux_target[-1] = target
        aux_target_tensor = torch.tensor(aux_target, dtype=torch.float32)
        
        return img, cont_tensor, cat_tensor, target_tensor, aux_target_tensor

## 2. 모델 아키텍처 (Cross-Attention Fusion)
- **SpatialCNN**: `(128, 17, 26)` Map 반환
- **CrossAttention**: GRU Query -> CNN Map
- **GatedFusion**: GRU + Attended Context 결합

In [ ]:
# [Helper] Self-Attention for CNN Refining
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        x_out = self.conv(x_cat)
        return self.sigmoid(x_out)

# [Modified] SpatialCNN (Retains Spatial Dimensions)
class SpatialCNN(nn.Module):
    def __init__(self):
        super(SpatialCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(2, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
        )
        self.sa = SpatialAttention()
        # No Pooling, No FC. Returns Feature Map directly.
        
    def forward(self, x):
        x = self.features(x)
        sa_map = self.sa(x)
        return x * sa_map # Shape: [B, 128, H/4, W/4]

class LSTMAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(LSTMAttention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)
    def forward(self, rnn_output):
        attn_weights = torch.softmax(self.attention(rnn_output), dim=1)
        return torch.sum(attn_weights * rnn_output, dim=1)

# [New] Cross Attention Module
class CrossAttention(nn.Module):
    def __init__(self, query_dim, key_dim, hidden_dim):
        super().__init__()
        self.query_proj = nn.Linear(query_dim, hidden_dim)
        self.key_proj = nn.Linear(key_dim, hidden_dim)
        self.value_proj = nn.Linear(key_dim, hidden_dim)
        self.scale = torch.sqrt(torch.tensor(hidden_dim, dtype=torch.float32))
        
    def forward(self, query, key_map):
        # query: [B, Q_Dim] (GRU State)
        # key_map: [B, K_Dim, H, W] (CNN Feature Map)
        
        B, C, H, W = key_map.shape
        # Flatten Key/Value: [B, H*W, C]
        flat_keys = key_map.view(B, C, -1).permute(0, 2, 1) 
        
        Q = self.query_proj(query).unsqueeze(1) # [B, 1, H_dim]
        K = self.key_proj(flat_keys)            # [B, H*W, H_dim]
        V = self.value_proj(flat_keys)          # [B, H*W, H_dim]
        
        # Attention Scores
        scores = torch.bmm(Q, K.transpose(1, 2)) / self.scale.to(query.device) # [B, 1, H*W]
        attn_weights = F.softmax(scores, dim=-1)
        
        # Context Vector
        context = torch.bmm(attn_weights, V) # [B, 1, H_dim]
        return context.squeeze(1)

# [Recycle] Gated Fusion (Same as V2.3)
class GatedFusion(nn.Module):
    def __init__(self, dim=128):
        super().__init__()
        self.gate_net = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.Sigmoid()
        )
    def forward(self, feat1, feat2):
        concat = torch.cat([feat1, feat2], dim=1)
        z = self.gate_net(concat)
        fused = z * feat1 + (1 - z) * feat2
        return fused

class MultiModalNetV8(nn.Module):
    def __init__(self, input_dim_cont, num_types, num_results, gru_hidden=96):
        super(MultiModalNetV8, self).__init__()
        # 1. Spatial CNN
        self.cnn = SpatialCNN() # Output: [B, 128, H, W]
        
        # 2. Sequential Embeddings
        self.type_emb = nn.Embedding(num_types, 8)
        self.result_emb = nn.Embedding(num_results, 8)
        total_input_dim = input_dim_cont + 8 + 8
        
        # 3. GRU (Split)
        self.gru_hidden = gru_hidden
        self.gru_fwd = nn.GRU(total_input_dim, gru_hidden, 2, batch_first=True, dropout=0.1)
        self.gru_bwd = nn.GRU(total_input_dim, gru_hidden, 2, batch_first=True, dropout=0.1)
        self.gru_attn = LSTMAttention(gru_hidden * 2)
        self.gru_fc = nn.Linear(gru_hidden * 2, 128)
        
        self.aux_fc = nn.Linear(gru_hidden, 2) 
        
        # 4. [NEW] Cross Attention
        # Query: GRU (128), Key: CNN (128)
        self.cross_attn = CrossAttention(query_dim=128, key_dim=128, hidden_dim=128)
        
        # 5. [Recycle] Gated Fusion
        # Fuse (Original GRU) + (Context-Aware Spatial Feature)
        self.gated_fusion = GatedFusion(dim=128)
        
        # 6. Final Head
        self.final_fc = nn.Sequential(
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, img, cont, cat, lengths):
        # CNN (Spatial Map)
        cnn_map = self.cnn(img) # [B, 128, 17, 26]
        
        # Embeddings
        emb_type = self.type_emb(cat[:, :, 0])
        emb_result = self.result_emb(cat[:, :, 1])
        x_seq = torch.cat([cont, emb_type, emb_result], dim=2)
        
        # GRU Forward/Backward
        packed_fwd = pack_padded_sequence(x_seq, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out_fwd_packed, _ = self.gru_fwd(packed_fwd)
        out_fwd, _ = pad_packed_sequence(out_fwd_packed, batch_first=True)
        
        x_seq_bwd = x_seq.clone()
        for i, length in enumerate(lengths):
            x_seq_bwd[i, :length] = x_seq[i, :length].flip(0)
        packed_bwd = pack_padded_sequence(x_seq_bwd, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out_bwd_packed, _ = self.gru_bwd(packed_bwd)
        out_bwd, _ = pad_packed_sequence(out_bwd_packed, batch_first=True)
        for i, length in enumerate(lengths):
            out_bwd[i, :length] = out_bwd[i, :length].flip(0)
            
        # GRU Feature
        gru_out_combined = torch.cat([out_fwd, out_bwd], dim=2)
        gru_ctx = self.gru_attn(gru_out_combined)
        seq_feat = F.relu(self.gru_fc(gru_ctx)) # [B, 128] (Intent)
        
        # [Cross-Attention Step]
        # Query: seq_feat (Intent)
        # Key/Value: cnn_map (Opportunity)
        context_feat = self.cross_attn(seq_feat, cnn_map) # [B, 128] (Relevant Space)
        
        # [Fusion Step]
        # Fuse Intent + Opportunity
        fused = self.gated_fusion(seq_feat, context_feat)
        
        final_out = self.final_fc(fused)
        aux_out = self.aux_fc(out_fwd)
        
        return final_out, aux_out

# Losses (Same)
class EuclideanLoss(nn.Module):
    def __init__(self):
        super(EuclideanLoss, self).__init__()
    def forward(self, pred, target):
        pred_real = pred * torch.tensor([105.0, 68.0], device=pred.device)
        target_real = target * torch.tensor([105.0, 68.0], device=target.device)
        return torch.mean(torch.sqrt(torch.sum((pred_real - target_real)**2, dim=1) + 1e-6))
        
class MaskedSeqEuclideanLoss(nn.Module):
    def __init__(self):
        super(MaskedSeqEuclideanLoss, self).__init__()
    def forward(self, pred, target, lengths):
        mask = torch.arange(pred.size(1), device=pred.device)[None, :] < lengths[:, None]
        mask = mask.unsqueeze(-1)
        pred_real = pred * torch.tensor([105.0, 68.0], device=pred.device)
        target_real = target * torch.tensor([105.0, 68.0], device=target.device)
        diff = pred_real - target_real
        dist = torch.sqrt(torch.sum(diff**2, dim=2) + 1e-6)
        dist = dist * mask.squeeze(-1)
        return dist.sum() / mask.sum()

## 3. 학습
- Models and Preprocessors saved in `models_final_v2_4/`

In [5]:
def multimodal_collate_fn(batch):
    imgs, conts, cats, targets, aux_targets = zip(*batch)
    imgs_batched = torch.stack(imgs, dim=0)
    lengths = torch.tensor([len(c) for c in conts], dtype=torch.long)
    conts_padded = pad_sequence(conts, batch_first=True)
    cats_padded = pad_sequence(cats, batch_first=True)
    targets = torch.stack(targets, dim=0)
    aux_targets_padded = pad_sequence(aux_targets, batch_first=True)
    return imgs_batched, conts_padded, cats_padded, lengths, targets, aux_targets_padded

def train_multimodal_v2_4(train_df, n_splits=10, epochs=100, batch_size=64, lr=0.001):
    gkf = GroupKFold(n_splits=n_splits)
    groups = train_df['game_id'] 
    
    fold_scores = []

    input_dim_cont = None
    num_types = None
    num_results = None
    
    for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, groups=groups)):
        fold_seed = FOLD_SEEDS[(fold % 10) + 1]
        seed_everything(fold_seed)
        print(f"[Fold {fold+1}] Seed fixed to {fold_seed}")
        print(f"\n=== Fold {fold+1}/{n_splits} ===")
        
        train_sub_df = train_df.iloc[train_idx].copy()
        val_sub_df = train_df.iloc[val_idx].copy()
        
        preprocessor = FootballPreprocessorMultimodal()
        preprocessor.fit(train_sub_df)

        if input_dim_cont is None:
            input_dim_cont = preprocessor.get_input_dim()
            num_types, num_results = preprocessor.get_num_classes()
            print(f"Model Dimensions: input={input_dim_cont}, types={num_types}, results={num_results}")
        
        train_episodes = preprocessor.transform(train_sub_df, is_train=True)
        val_episodes = preprocessor.transform(val_sub_df, is_train=True)
        
        train_dataset = MultiModalDataset(train_episodes, augment=True, cache_images=True)
        val_dataset = MultiModalDataset(val_episodes, augment=False, cache_images=True)
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=multimodal_collate_fn, drop_last=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=multimodal_collate_fn)
        
        # [V2.4 Model]
        model = MultiModalNetV8(input_dim_cont, num_types, num_results).to(DEVICE)
        
        criterion_main = EuclideanLoss()
        criterion_aux = MaskedSeqEuclideanLoss() 
        
        optimizer = optim.Adam(model.parameters(), lr=lr) 
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
        
        best_dist = float('inf')
        best_state = None
        patience_counter = 0
        patience_limit = 15
        
        if not os.path.exists('models_final_v2_4'):
            os.makedirs('models_final_v2_4')
            
        for epoch in range(epochs):
            model.train()
            train_loss = 0
            for batch in train_loader:
                imgs, cont, cat, lengths, target, aux_target = batch
                imgs, cont, cat, lengths, target, aux_target = imgs.to(DEVICE), cont.to(DEVICE), cat.to(DEVICE), lengths.to(DEVICE), target.to(DEVICE), aux_target.to(DEVICE)
                
                optimizer.zero_grad()
                pred, aux_pred = model(imgs, cont, cat, lengths)
                
                loss_m = criterion_main(pred, target)
                loss_a = criterion_aux(aux_pred, aux_target, lengths)
                
                loss = loss_m + 0.5 * loss_a
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            
            train_loss /= len(train_loader)
            
            model.eval()
            val_dists = []
            with torch.no_grad():
                for batch in val_loader:
                    imgs, cont, cat, lengths, target, aux_target = batch
                    imgs, cont, cat, lengths, target, aux_target = imgs.to(DEVICE), cont.to(DEVICE), cat.to(DEVICE), lengths.to(DEVICE), target.to(DEVICE), aux_target.to(DEVICE)
                    
                    pred, _ = model(imgs, cont, cat, lengths)
                    
                    pred_real = pred.cpu().numpy() * np.array([105.0, 68.0])
                    target_real = target.cpu().numpy() * np.array([105.0, 68.0])
                    val_dists.extend(np.sqrt(np.sum((pred_real - target_real)**2, axis=1)))
            
            mean_dist = np.mean(val_dists)
            scheduler.step(mean_dist)
            print(f"Epoch {epoch+1}: Train Loss {train_loss:.4f}, Val Dist {mean_dist:.4f}")
            
            if mean_dist < best_dist:
                best_dist = mean_dist
                best_state = model.state_dict()
                torch.save(best_state, f"models_final_v2_4/multimodal_v2_4_{fold+1}.pth")
                joblib.dump(preprocessor, f"models_final_v2_4/preprocessor_v2_4_{fold+1}.pkl")
                print(f"  -> Saved Best Model (Dist: {best_dist:.4f})")
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience_limit:
                    print("Early Stopping")
                    break
        
        print(f"Fold {fold+1} Finished. Best Valid Dist: {best_dist:.4f}")
        fold_scores.append(best_dist)
    
    with open('scores_v2_4.json', 'w') as f:
        json.dump(fold_scores, f)
        
    print(f"Average Score: {np.mean(fold_scores):.4f}")

train_multimodal_v2_4(train_df)

[Fold 1] Seed fixed to 42

=== Fold 1/10 ===
Model Dimensions: input=10, types=27, results=21
Pre-rendering 13827 images (Cache Enabled)...


Caching Images: 100%|██████████| 13827/13827 [00:00<00:00, 16078.15it/s]


Pre-rendering 1601 images (Cache Enabled)...


Caching Images: 100%|██████████| 1601/1601 [00:00<00:00, 16121.29it/s]


Epoch 1: Train Loss 66.7936, Val Dist 20.3563
  -> Saved Best Model (Dist: 20.3563)
Epoch 2: Train Loss 48.8862, Val Dist 18.3855
  -> Saved Best Model (Dist: 18.3855)
Epoch 3: Train Loss 46.8768, Val Dist 17.7056
  -> Saved Best Model (Dist: 17.7056)
Epoch 4: Train Loss 45.2381, Val Dist 15.9559
  -> Saved Best Model (Dist: 15.9559)
Epoch 5: Train Loss 44.2525, Val Dist 16.1914
Epoch 6: Train Loss 43.3789, Val Dist 16.3165
Epoch 7: Train Loss 42.8640, Val Dist 15.0756
  -> Saved Best Model (Dist: 15.0756)
Epoch 8: Train Loss 42.3256, Val Dist 14.9966
  -> Saved Best Model (Dist: 14.9966)
Epoch 9: Train Loss 41.9273, Val Dist 14.7509
  -> Saved Best Model (Dist: 14.7509)
Epoch 10: Train Loss 41.6915, Val Dist 15.2310
Epoch 11: Train Loss 41.4397, Val Dist 14.8034
Epoch 12: Train Loss 41.0741, Val Dist 14.6897
  -> Saved Best Model (Dist: 14.6897)
Epoch 13: Train Loss 40.9516, Val Dist 14.7397
Epoch 14: Train Loss 40.6653, Val Dist 14.2535
  -> Saved Best Model (Dist: 14.2535)
Epoch 15:

...

Epoch 1: Train Loss 66.6367, Val Dist 19.1730
  -> Saved Best Model (Dist: 19.1730)
Epoch 2: Train Loss 49.2084, Val Dist 17.5960
  -> Saved Best Model (Dist: 17.5960)
Epoch 3: Train Loss 46.4524, Val Dist 17.0664
  -> Saved Best Model (Dist: 17.0664)
Epoch 4: Train Loss 45.1820, Val Dist 16.5760
  -> Saved Best Model (Dist: 16.5760)
Epoch 5: Train Loss 44.2239, Val Dist 16.9627
Epoch 6: Train Loss 43.3594, Val Dist 16.0995
  -> Saved Best Model (Dist: 16.0995)
Epoch 7: Train Loss 42.7619, Val Dist 15.4788
  -> Saved Best Model (Dist: 15.4788)
Epoch 8: Train Loss 42.1984, Val Dist 15.3503
  -> Saved Best Model (Dist: 15.3503)
Epoch 9: Train Loss 41.8739, Val Dist 15.5720
Epoch 10: Train Loss 41.6344, Val Dist 15.5810
Epoch 11: Train Loss 41.2235, Val Dist 15.1337
  -> Saved Best Model (Dist: 15.1337)
Epoch 12: Train Loss 41.0454, Val Dist 14.8450
  -> Saved Best Model (Dist: 14.8450)
Epoch 13: Train Loss 40.7777, Val Dist 15.0255
Epoch 14: Train Loss 40.5892, Val Dist 14.9094
Epoch 15:

Caching Images: 100%|██████████| 13850/13850 [00:00<00:00, 15916.16it/s]


Pre-rendering 1578 images (Cache Enabled)...


Caching Images: 100%|██████████| 1578/1578 [00:00<00:00, 15154.79it/s]


Epoch 1: Train Loss 65.1114, Val Dist 20.7604
  -> Saved Best Model (Dist: 20.7604)
Epoch 2: Train Loss 49.1161, Val Dist 17.3854
  -> Saved Best Model (Dist: 17.3854)
Epoch 3: Train Loss 46.4268, Val Dist 16.5067
  -> Saved Best Model (Dist: 16.5067)
Epoch 4: Train Loss 45.1800, Val Dist 16.1971
  -> Saved Best Model (Dist: 16.1971)
Epoch 5: Train Loss 44.1621, Val Dist 14.9373
  -> Saved Best Model (Dist: 14.9373)
Epoch 6: Train Loss 43.3442, Val Dist 15.3802
Epoch 7: Train Loss 42.7860, Val Dist 15.3025
Epoch 8: Train Loss 42.3702, Val Dist 14.8350
  -> Saved Best Model (Dist: 14.8350)
Epoch 9: Train Loss 41.8607, Val Dist 14.6995
  -> Saved Best Model (Dist: 14.6995)
Epoch 10: Train Loss 41.5722, Val Dist 14.9319
Epoch 11: Train Loss 41.4076, Val Dist 14.9642
Epoch 12: Train Loss 41.1753, Val Dist 14.5167
  -> Saved Best Model (Dist: 14.5167)
Epoch 13: Train Loss 40.8853, Val Dist 14.3068
  -> Saved Best Model (Dist: 14.3068)
Epoch 14: Train Loss 40.6700, Val Dist 14.5405
Epoch 15:

In [6]:
# Inference V2.4
class TestMMDataset(Dataset):
    def __init__(self, sub_df, preprocessor, img_size=(68, 105)):
        self.sub_df = sub_df
        self.prep = preprocessor
        self.H, self.W = img_size
    def __len__(self): return len(self.sub_df)
    def _gen_img(self, cont):
        img = np.zeros((2, self.H, self.W), dtype=np.float32)
        if len(cont) == 0: return torch.tensor(img, dtype=torch.float32)
        xs, ys = cont[:,0], cont[:,1]
        xi = (xs*(self.W-1)).round().astype(int).clip(0, self.W-1)
        yi = (ys*(self.H-1)).round().astype(int).clip(0, self.H-1)
        for t in range(len(cont)):
            img[0, yi[t], xi[t]] = 1.0
            decay = (t+1)/len(cont)
            if t > 0:
                cv2.line(img[1], (xi[t-1], yi[t-1]), (xi[t], yi[t]), color=decay, thickness=1)
            else:
                img[1, yi[t], xi[t]] = decay
        return torch.tensor(img, dtype=torch.float32)
    def __getitem__(self, idx):
        path = self.sub_df.iloc[idx]['path']
        full_path = os.path.join("data", path[2:] if path.startswith("./") else path)
        try:
            df = pd.read_csv(full_path)
            eps = self.prep.transform(df, False)
        except: eps = []
        if not eps:
            d = self.prep.get_input_dim()
            c = torch.zeros((1,d), dtype=torch.float32)
            cat = torch.zeros((1,5), dtype=torch.long)
            img = torch.zeros((2, self.H, self.W), dtype=torch.float32)
        else:
            data = eps[0]
            c = torch.tensor(data['cont'], dtype=torch.float32)
            cat = torch.tensor(data['cat'], dtype=torch.long)
            img = self._gen_img(data['cont'])
        return img, c, cat

def test_collate(b):
    img, c, cat = zip(*b)
    ib = torch.stack(img)
    l = torch.tensor([len(x) for x in c], dtype=torch.long)
    cp = pad_sequence(c, batch_first=True)
    catp = pad_sequence(cat, batch_first=True)
    return ib, cp, catp, l

sub = pd.read_csv("data/sample_submission.csv")
tm = pd.read_csv("data/test.csv")
sub = sub.merge(tm, on="game_episode", how="left")

# Load Ref for dimensions
prep_ref = joblib.load('models_final_v2_4/preprocessor_v2_4_1.pkl')
GD = prep_ref.get_input_dim()
nT, nR = prep_ref.get_num_classes()

all_preds = []
for f in range(1, 11):
    try:
        prep = joblib.load(f'models_final_v2_4/preprocessor_v2_4_{f}.pkl')
        ds = TestMMDataset(sub, prep)
        dl = DataLoader(ds, 128, False, collate_fn=test_collate)
        model = MultiModalNetV8(GD, nT, nR).to(DEVICE)
        model.load_state_dict(torch.load(f'models_final_v2_4/multimodal_v2_4_{f}.pth', map_location=DEVICE))
        model.eval()
        fold_p = []
        with torch.no_grad():
            for b in tqdm(dl, desc=f'Fold {f}'):
                i, c, ct, l = [x.to(DEVICE) for x in b]
                p, _ = model(i, c, ct, l)
                pn = p.cpu().numpy()
                pn[:,0] *= 105; pn[:,1] *= 68
                fold_p.append(pn)
        all_preds.append(np.vstack(fold_p))
    except Exception as e: print(e)

if all_preds:
    avg = np.mean(all_preds, axis=0)
    sub['end_x'] = avg[:,0].clip(0, 105)
    sub['end_y'] = avg[:,1].clip(0, 68)
    sub[['game_episode','end_x','end_y']].to_csv('Final_env_v2_4.csv', index=False)
    print("Done: Final_env_v2_4.csv")

C:\Users\syt07\AppData\Local\Temp\ipykernel_18392\1272777022.py:65: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'models_final_v2_4/multim

Done: Final_env_v2_4.csv
